# 房价预测分析综合项目

## 项目概述
本项目通过完整的机器学习流程，使用加州房价数据集（California Housing Dataset）进行房价预测分析。项目涵盖数据探索、特征工程、模型训练、评估优化和结果可视化全流程，旨在帮助用户掌握从数学原理到AI应用的完整技能链条。

### 数据集介绍
加州房价数据集包含20640个样本，每个样本有8个特征：
1. MedInc：该街区居民收入中位数
2. HouseAge：该街区房屋使用年数中位数
3. AveRooms：该街区平均房间数
4. AveBedrms：该街区平均卧室数
5. Population：该街区人口数
6. AveOccup：平均入住率
7. Latitude：纬度
8. Longitude：经度

目标变量：房屋价值中位数（单位：十万美元）

## 1. 环境准备与数据加载

In [ ]:
# 导入所需库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK JP']
plt.rcParams['axes.unicode_minus'] = False

# 设置样式
sns.set_style("whitegrid")
plt.style.use('seaborn-v0_8-talk')

In [ ]:
# 加载数据集
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseVal')

print(f"数据集形状: X={X.shape}, y={y.shape}")
print("\n特征名称:")
for i, feature in enumerate(housing.feature_names):
    print(f"  {i+1}. {feature}")

print("\n前5行数据:")
X.head()

## 2. 探索性数据分析（EDA）

In [ ]:
# 目标变量分布
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(y, kde=True, bins=50)
plt.title('房屋价值中位数分布')
plt.xlabel('房屋价值中位数（十万美元）')
plt.ylabel('频数')

plt.subplot(1, 2, 2)
sns.boxplot(y=y)
plt.title('房屋价值中位数箱线图')
plt.xlabel('房屋价值中位数（十万美元）')

plt.tight_layout()
plt.show()

In [ ]:
# 特征分布
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i, feature in enumerate(X.columns):
    sns.histplot(X[feature], kde=True, bins=50, ax=axes[i])
    axes[i].set_title(f'{feature}分布')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('频数')

plt.tight_layout()
plt.show()

In [ ]:
# 特征与目标变量的关系
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i, feature in enumerate(X.columns):
    sns.scatterplot(x=X[feature], y=y, alpha=0.5, ax=axes[i])
    axes[i].set_title(f'{feature} vs 房屋价值')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('房屋价值中位数（十万美元）')

plt.tight_layout()
plt.show()

In [ ]:
# 相关系数矩阵
plt.figure(figsize=(10, 8))
corr_matrix = pd.concat([X, y], axis=1).corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('特征与目标变量相关系数矩阵')
plt.show()

# 显示与目标变量的相关性排序
corr_with_target = corr_matrix['MedHouseVal'].sort_values(ascending=False)
print("特征与目标变量相关性排序:")
for feature, corr in corr_with_target.items():
    if feature != 'MedHouseVal':
        print(f"  {feature}: {corr:.3f}")

## 3. 数据预处理

In [ ]:
# 检查缺失值
print("缺失值检查:")
print(f"X缺失值总数: {X.isnull().sum().sum()}")
print(f"y缺失值总数: {y.isnull().sum()}")

# 检查异常值（使用Z-score方法）
from scipy import stats
z_scores = np.abs(stats.zscore(X))
outliers = (z_scores > 3).sum(axis=1) > 0
print(f"\n异常值数量（Z-score > 3）: {outliers.sum()} ({outliers.sum()/len(X)*100:.2f}%)")

# 分割训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\n训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

In [ ]:
# 特征标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 转换为DataFrame
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

print("标准化后的训练集前5行:")
X_train_scaled_df.head()

## 4. 特征工程

In [ ]:
# 创建多项式特征
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

print(f"原始特征数量: {X_train_scaled.shape[1]}")
print(f"多项式特征后特征数量: {X_train_poly.shape[1]}")

# 创建交互特征（手动创建一些有意义的交互特征）
X_train_engineered = X_train_scaled_df.copy()
X_test_engineered = X_test_scaled_df.copy()

# 收入与房屋年龄的交互
X_train_engineered['MedInc_HouseAge'] = X_train_engineered['MedInc'] * X_train_engineered['HouseAge']
X_test_engineered['MedInc_HouseAge'] = X_test_engineered['MedInc'] * X_test_engineered['HouseAge']

# 平均房间数与平均卧室数的比率
X_train_engineered['Rooms_Bedrms_Ratio'] = X_train_engineered['AveRooms'] / (X_train_engineered['AveBedrms'] + 1e-6)
X_test_engineered['Rooms_Bedrms_Ratio'] = X_test_engineered['AveRooms'] / (X_test_engineered['AveBedrms'] + 1e-6)

# 人口密度（人口/入住率）
X_train_engineered['Population_Density'] = X_train_engineered['Population'] / (X_train_engineered['AveOccup'] + 1e-6)
X_test_engineered['Population_Density'] = X_test_engineered['Population'] / (X_test_engineered['AveOccup'] + 1e-6)

print(f"\n特征工程后特征数量: {X_train_engineered.shape[1]}")
print("\n新增特征:")
print("1. MedInc_HouseAge: 收入与房屋年龄交互项")
print("2. Rooms_Bedrms_Ratio: 房间与卧室比率")
print("3. Population_Density: 人口密度")

# 显示前5行
X_train_engineered.head()

## 5. 模型训练与评估

In [ ]:
# 定义评估函数
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """评估模型并返回指标"""
    # 训练
    model.fit(X_train, y_train)
    
    # 预测
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # 计算指标
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    return {
        '模型': model_name,
        '训练集MSE': train_mse,
        '测试集MSE': test_mse,
        '训练集MAE': train_mae,
        '测试集MAE': test_mae,
        '训练集R²': train_r2,
        '测试集R²': test_r2
    }

In [ ]:
# 训练多个模型
models = {
    '线性回归': LinearRegression(),
    '岭回归': Ridge(alpha=1.0),
    'Lasso回归': Lasso(alpha=0.1),
    '决策树': DecisionTreeRegressor(max_depth=5, random_state=42),
    '随机森林': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    '梯度提升': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

# 评估结果存储
results = []

# 使用特征工程后的数据
for name, model in models.items():
    result = evaluate_model(model, X_train_engineered, X_test_engineered, y_train, y_test, name)
    results.append(result)

# 显示结果
results_df = pd.DataFrame(results)
results_df

In [ ]:
# 可视化模型性能
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R²分数
axes[0].bar(results_df['模型'], results_df['测试集R²'], color='skyblue')
axes[0].set_title('各模型测试集R²分数')
axes[0].set_xlabel('模型')
axes[0].set_ylabel('R²分数')
axes[0].tick_params(axis='x', rotation=45)

# MSE
axes[1].bar(results_df['模型'], results_df['测试集MSE'], color='lightcoral')
axes[1].set_title('各模型测试集MSE')
axes[1].set_xlabel('模型')
axes[1].set_ylabel('MSE')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 预测结果可视化
best_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
best_model.fit(X_train_engineered, y_train)
y_pred = best_model.predict(X_test_engineered)

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('真实值')
plt.ylabel('预测值')
plt.title('预测值 vs 真实值（随机森林）')
plt.grid(True)
plt.show()

# 残差分析
residuals = y_test - y_pred
plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('预测值')
plt.ylabel('残差')
plt.title('残差分析')
plt.grid(True)
plt.show()

## 6. 特征重要性分析

In [ ]:
# 随机森林特征重要性
feature_importance = pd.DataFrame({
    '特征': X_train_engineered.columns,
    '重要性': best_model.feature_importances_
}).sort_values('重要性', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='重要性', y='特征', data=feature_importance)
plt.title('随机森林特征重要性')
plt.xlabel('重要性')
plt.ylabel('特征')
plt.show()

print("特征重要性排序:")
for i, row in feature_importance.iterrows():
    print(f"  {i+1}. {row['特征']}: {row['重要性']:.4f}")

## 7. 模型优化（超参数调优）

In [ ]:
# 随机森林超参数调优
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)
grid_search.fit(X_train_engineered, y_train)

print(f"最佳参数: {grid_search.best_params_}")
print(f"最佳交叉验证分数（负MSE）: {grid_search.best_score_:.4f}")

# 使用最佳模型
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test_engineered)

best_mse = mean_squared_error(y_test, y_pred_best)
best_r2 = r2_score(y_test, y_pred_best)

print(f"\n优化后模型性能:")
print(f"测试集MSE: {best_mse:.4f}")
print(f"测试集R²: {best_r2:.4f}")

## 8. 项目总结

In [ ]:
# 性能对比
print("=== 模型性能总结 ===")
print("\n1. 各模型测试集R²分数:")
for _, row in results_df.iterrows():
    print(f"   {row['模型']}: {row['测试集R²']:.4f}")

print("\n2. 最佳模型（随机森林）:")
print(f"   优化前R²: {results_df[results_df['模型']=='随机森林']['测试集R²'].values[0]:.4f}")
print(f"   优化后R²: {best_r2:.4f}")

print("\n3. 关键发现:")
print("   • 收入中位数（MedInc）是最重要的预测特征")
print("   • 地理位置（经纬度）对房价有显著影响")
print("   • 特征工程（交互特征）提升了模型性能")
print("   • 集成方法（随机森林、梯度提升）优于线性模型")

print("\n4. 项目收获:")
print("   • 掌握了完整的数据科学工作流程")
print("   • 理解了不同回归算法的数学原理")
print("   • 学会了特征工程和模型优化的技巧")
print("   • 培养了结果分析和可视化能力")